# 03 — Vibe Architect Studio (Mixtape Edition): Napkin Sketch → Real BIM → Photoreal Render

Built for **architects, designers, and vibe coders** — run all cells once to launch a **Gradio Studio App**!

### What's New in the Mixtape Edition
1. **Drop a napkin sketch / Procreate plan** (or click **🎲 Drop Synthetic Napkin Sketch** to test instantly) OR type a vibe prompt — OR click **⚡ Load Showcase Building** for zero-LLM-wait instant BIM + renders!
2. **Real MutinyBIM Exports:** Crisp **Architectural Floor Plan SVGs** (`sceneToSvg`), **Vector PDF Sheets** (`bim-sheets`), **IFC 4.3** (`exportIfc`), interactive **3D GLB**, and a **360° Isometric Orbit Turntable GIF**.
3. **Geometry-Locked Photoreal Render (`SDXL-Depth` or `FLUX.1-Depth`):** Depth-conditioned from the real IFC geometry so the render follows the model's walls and openings.
4. **🎛️ Mixtape A/B Strip:** One click renders a 3-track comparison triptych across art directions or weight-space LoRA faders.
5. **🎬 4-Shot Presentation Storyboard:** Renders Isometric Cutaway + Eye-Level Interior + Opposite Interior + Exterior Corner on a single `2×2` presentation board.
6. **👁️ Critique & Redraw:** Lets Gemma 4's native vision inspect its own floor plan + 3D cutaway and redesign automatically.
7. **☀️ Solar Dome & Direct Sun Hours:** Geolocation + solstice/equinox Sun-Dial HUD, Ladybug-style 3D sky dome, GPU/NumPy Direct Sun Hours heatmap (`ScalarHeatmapOverlay` JSON), and hourly shadow-sweep GIF.

### Hardware & Credit Budget
* **L4 (24GB VRAM, ~1.6 CCUs/hr):** Sweet spot — runs 4-bit Gemma 4 + CPU-offloaded SDXL-Depth ControlNet cleanly.
* **A100 (40GB / 80GB):** Faster multi-shot storyboards & optional `FLUX.1-Depth-dev`.


In [ ]:
#@title 1. Check GPU, Clone Repos, Install Node 22, and Boot Warm MutinyBIM Export Server
!nvidia-smi

import json, subprocess, sys, time
from pathlib import Path

MIXER_REPO_URL = "https://github.com/5hyycjb77y/mutinybim-mixer.git" #@param {type:"string"}
WORK_DIR = Path("/content/work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
%cd /content/work

if not (WORK_DIR / "mutinybim-mixer").exists():
    !git clone --depth 1 {MIXER_REPO_URL} mutinybim-mixer || true

MIXER_DIR = WORK_DIR / "mutinybim-mixer"
assert MIXER_DIR.exists(), "Upload mutinybim-mixer to /content/work/mutinybim-mixer or set MIXER_REPO_URL."

SPEC = json.loads((MIXER_DIR / "spec.json").read_text())
REGISTRY = json.loads((MIXER_DIR / "registry.json").read_text())
MUTINY_DIR = WORK_DIR / "mutinybim"

if not MUTINY_DIR.exists():
    !git clone {SPEC["mutinybim"]["repo"]} {MUTINY_DIR}
!git -C {MUTINY_DIR} fetch --depth 1 origin {SPEC["mutinybim"]["commit"]} && git -C {MUTINY_DIR} checkout {SPEC["mutinybim"]["commit"]}

node_v = subprocess.run(["node", "-v"], capture_output=True, text=True)
if node_v.returncode != 0 or int(node_v.stdout.strip().lstrip("v").split(".")[0]) < 22:
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y nodejs
!node -v

if not (MUTINY_DIR / "node_modules").exists():
    !npm --prefix {MUTINY_DIR} install --no-audit --no-fund

verifier_log = open("/content/verifier.log", "w")
verifier_proc = subprocess.Popen(
    ["node", str(MIXER_DIR / "verifier/server.mjs"), "--mutinybim", str(MUTINY_DIR), "--port", "8787"],
    stdout=verifier_log,
    stderr=subprocess.STDOUT,
)

sys.path.insert(0, str(MIXER_DIR))
from verifier.verify_client import MutinyVerifierClient

client = MutinyVerifierClient("http://127.0.0.1:8787", timeout=60.0)
for _ in range(30):
    try:
        print("MutinyBIM Verifier & Exporter ready:", client.health())
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Verifier failed to start — inspect /content/verifier.log")


In [ ]:
#@title 2. Install Studio Python Dependencies (Gradio, Unsloth, Diffusers, IfcOpenShell, Trimesh, CairoSVG)
!pip install -q --upgrade pip
!pip install -q gradio unsloth diffusers transformers accelerate peft safetensors ifcopenshell trimesh cairosvg pillow matplotlib huggingface_hub


In [ ]:
#@title 3. Studio Configuration
RENDER_BACKEND = "sdxl_depth" #@param ["sdxl_depth", "flux_depth"]
ENABLE_VISION_SKETCH = True #@param {type:"boolean"}
ENABLE_THINKING = True #@param {type:"boolean"}
COMMUNITY_LORA_DIR = "" #@param {type:"string"}
RENDER_STEPS = 24 #@param {type:"integer"}
SHARE_PUBLIC_GRADIO_LINK = False #@param {type:"boolean"}

STUDIO_OUT = Path("/content/studio_runs")
STUDIO_OUT.mkdir(parents=True, exist_ok=True)
print(f"Studio ready | backend={RENDER_BACKEND} | thinking={ENABLE_THINKING} | custom_lora={COMMUNITY_LORA_DIR or 'base Gemma 4 31B'}")


In [ ]:
#@title 4. Load Gemma 4 31B (+ Optional Community Trade LoRA) & Define Self-Repair Generator
import torch
from unsloth import FastModel
from peft import PeftModel
from mixer.prompts import SYSTEM_PROMPT_TEMPLATE
from verifier.verify_client import extract_typescript

REF_BUNDLE = client.reference(compact=True)
KIT_REF_MD = REF_BUNDLE["markdown"]
OFFICIAL_EXAMPLES = {ex["name"]: ex for ex in REF_BUNDLE.get("examples", [])}

base_model, tokenizer = FastModel.from_pretrained(
    model_name=SPEC["base_model"]["unsloth_id"],
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

if COMMUNITY_LORA_DIR.strip():
    model = PeftModel.from_pretrained(base_model, COMMUNITY_LORA_DIR.strip(), adapter_name="studio_active")
    print("Attached community LoRA:", COMMUNITY_LORA_DIR)
else:
    model = base_model
    print("Running base unsloth/gemma-4-31B-it (no custom LoRA required).")

FastModel.for_inference(model)


def _gemma_generate(messages, images=None, max_new_tokens=1800, temperature=0.7):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    try:
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=ENABLE_THINKING,
        )
    except TypeError:
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    if images:
        try:
            inputs = tokenizer(text=[prompt_text], images=images, return_tensors="pt").to(model.device)
        except Exception:
            inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    else:
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    gen_cfg = SPEC["base_model"]["generation_defaults"]
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=gen_cfg["top_p"],
            top_k=gen_cfg["top_k"],
            do_sample=True,
        )
    return tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)


def apply_studio_faders(spatial_rigor: float, mep_detail: float, clash_polish: float) -> str:
    adapters = REGISTRY.get("adapters", [])
    if not adapters:
        return f"Base Gemma 4 active (Faders noted: Spatial={spatial_rigor:.1f}, MEP={mep_detail:.1f}, ClashNeg={clash_polish:.1f})"
    try:
        from mixer.lora_ops import load_lora_deltas, merge_deltas, refactorize_deltas_to_lora, save_merged_lora
        deltas_list, weights, cfgs = [], [], []
        for entry in adapters:
            d, cfg = load_lora_deltas(entry["hf_repo"], revision=entry.get("revision"))
            trade = entry["trade"]
            if trade in ("architecture", "structural"):
                w = spatial_rigor
            elif trade in ("mechanical", "plumbing", "electrical"):
                w = mep_detail
            elif trade == "clash_negation":
                w = -abs(clash_polish)
            else:
                w = float(entry.get("default_weight", 0.7))
            deltas_list.append(d)
            weights.append(w)
            cfgs.append(cfg)
        merged = merge_deltas(deltas_list, weights, method="dare_ties", density=0.5, drop_rate=0.2)
        lora_sd, _ = refactorize_deltas_to_lora(merged, out_rank=16, lora_alpha=16)
        blend_dir = save_merged_lora(lora_sd, cfgs[0], STUDIO_OUT / "live_blend")
        if hasattr(model, "load_adapter"):
            if "studio_blend" in getattr(model, "peft_config", {}):
                model.delete_adapter("studio_blend")
            model.load_adapter(str(blend_dir), adapter_name="studio_blend")
            model.set_adapter("studio_blend")
        return f"Weight-space blended {len(adapters)} trade LoRAs!"
    except Exception as exc:
        return f"Fader blend fallback ({exc})"


def generate_and_export_bim(vibe_prompt: str, sketch_image=None, critique_context: str = "", critique_images=None):
    sys_msg = (
        SYSTEM_PROMPT_TEMPLATE
        + "\nAlways place at least one plan sheet (`documents.sheet` + `documents.planView` + `documents.place`) "
        + "so architectural drawings render cleanly.\n\n## MutinyBIM Script Kit Reference\n"
        + KIT_REF_MD[:4800]
    )

    user_parts = []
    images = []
    if sketch_image is not None and ENABLE_VISION_SKETCH:
        images.append(sketch_image)
        user_parts.append({"type": "image", "image": sketch_image})
    if critique_images:
        for img in critique_images:
            images.append(img)
            user_parts.append({"type": "image", "image": img})

    text_prompt = (
        f"Design Request:\n{vibe_prompt}\n"
        + (f"\nArt Director Critique & Improvement Instructions:\n{critique_context}\n" if critique_context else "")
        + "\nReturn one complete runnable ```ts block."
    )
    if images:
        user_parts.append({"type": "text", "text": text_prompt})
        user_content = user_parts
    else:
        user_content = text_prompt

    messages = [
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": user_content},
    ]

    bundle = None
    ts_source = ""
    for attempt in range(2):
        raw_out = _gemma_generate(messages, images=images if attempt == 0 else None)
        ts_source = extract_typescript(raw_out)
        bundle = client.export_bundle(ts_source, project_name="Vibe Architect Studio", start="empty")
        if bundle.verify.ok and bundle.ifc_text:
            break
        if attempt == 0:
            err_msg = bundle.verify.error or bundle.verify.stage_failed or "Script failed verification."
            messages.append({"role": "assistant", "content": f"```ts\n{ts_source}\n```"})
            messages.append({
                "role": "user",
                "content": f"MutinyBIM verifier reported ({bundle.verify.stage_failed}): {err_msg}. Fix the TypeScript script and output one clean ```ts block.",
            })

    return ts_source, bundle


In [ ]:
#@title 5. Studio Mixtape Pipelines (Generate, Showcase, Sketch Drop, Mixtape Strip, 4-Shot Board & 360° Orbit)
import base64
import time
from PIL import Image
from mixer.sketch_engine import (
    SKETCH_STYLES,
    SKETCH_TOPOLOGIES,
    generate_sketch_sample,
    render_parametric_sketch_png,
)
from mixer.solar_engine import (
    CITY_PRESETS,
    KEY_DATES,
    compute_direct_sun_hours,
    compute_incident_solar_radiation,
    read_ifc_true_north_deg,
    render_incident_radiation_png,
    render_solar_control_frame,
    render_solar_dome_png,
    render_solstice_shadow_gif,
    render_sun_dial_svg,
)
from mixer.studio_render import (
    ART_DIRECTION_PRESETS,
    CAMERA_PRESETS,
    MIXTAPE_PRESETS,
    VIBE_INSPIRATION_CHIPS,
    compose_grid,
    ifc_to_glb_and_control,
    render_ifc_turntable_gif,
    render_photoreal,
    render_solar_photoreal_timelapse,
    render_storyboard_4up,
    svg_to_png,
)


def _format_sheets_html(sheets: list[dict]) -> str:
    if not sheets:
        return "<div style='padding:24px;color:#666;'>No sheets were emitted by this script. Try asking for an architectural plan sheet (A-101).</div>"
    cards = []
    for sh in sheets:
        cards.append(
            f"<div style='margin-bottom:20px;border:1px solid #ddd;border-radius:10px;padding:12px;background:#fff;'>"
            f"<div style='font-weight:600;margin-bottom:8px;color:#111;'>Sheet {sh['number']} — {sh['title']}</div>"
            f"<div style='width:100%;overflow:auto;background:#fafafa;border:1px solid #eee;'>{sh['svg']}</div>"
            f"</div>"
        )
    return "\n".join(cards)


def _build_visuals_from_bundle(
    ts_source: str,
    bundle,
    vibe_prompt: str,
    style_key: str,
    camera_key: str,
    fader_msg: str,
    run_dir: Path,
    seed: int,
    *,
    make_turntable: bool = True,
    banner_prefix: str = "",
):
    run_dir.mkdir(parents=True, exist_ok=True)
    ts_path = run_dir / "model.ts"
    ts_path.write_text(ts_source, encoding="utf-8")

    ifc_path = pdf_path = glb_path = depth_path = clay_path = photoreal_path = turntable_path = None
    model_tn = 0.0
    geom: dict = {}
    if bundle.ifc_text:
        ifc_path = str(run_dir / "model.ifc")
        Path(ifc_path).write_text(bundle.ifc_text, encoding="utf-8")
        model_tn = read_ifc_true_north_deg(ifc_path)
        geom = ifc_to_glb_and_control(ifc_path, run_dir, camera_preset=camera_key)
        glb_path, depth_path, clay_path = geom["glb_path"], geom["depth_path"], geom["clay_path"]
        if depth_path:
            photoreal_path = render_photoreal(
                depth_path,
                vibe_prompt,
                style_key,
                camera_key,
                run_dir / "render.png",
                backend=RENDER_BACKEND,
                steps=RENDER_STEPS,
                seed=int(seed),
            )
        if make_turntable:
            try:
                turntable_path = render_ifc_turntable_gif(ifc_path, run_dir / "turntable.gif", num_frames=18)
            except Exception:
                turntable_path = None

    if bundle.pdf_base64:
        pdf_path = str(run_dir / "sheets.pdf")
        Path(pdf_path).write_bytes(base64.b64decode(bundle.pdf_base64))

    total_elems = sum(bundle.verify.kinds.values())
    clashes = int(bundle.verify.lint_summary.get("errors", 0))
    warnings = int(bundle.verify.lint_summary.get("warnings", 0))
    lint_score = float(bundle.verify.breakdown.get("bim_lint_clean", 0.0))
    badge = "✅" if bundle.verify.ok else "⚠️"
    prefix = f"{banner_prefix} · " if banner_prefix else ""
    status_md = (
        f"### {badge} {prefix}Overall Score: `{bundle.verify.reward:.2f}` · "
        f"Lint Score: `{lint_score:.2f}` (`{clashes}` Errors / `{warnings}` Warnings) · "
        f"`{total_elems}` BIM Elements · `{len(bundle.sheets)}` Sheets\n"
        f"*{fader_msg}* · Kinds: `{json.dumps(bundle.verify.kinds)}`"
    )
    if bundle.verify.error:
        status_md += f"\n\n> Note: `{bundle.verify.error}`"
    if geom.get("camera_fallback"):
        status_md += (
            f"\n\n> ⚠️ **Camera fallback (`{geom.get('fallback_reason')}`)** — "
            f"forward wall clearance: `{geom.get('forward_hit_m')} m`"
        )

    sheets_html = _format_sheets_html(bundle.sheets)
    downloads = [p for p in (ifc_path, pdf_path, glb_path, turntable_path, photoreal_path, str(ts_path)) if p]
    state = {
        "vibe_prompt": vibe_prompt,
        "ts_source": ts_source,
        "ifc_path": ifc_path,
        "depth_path": depth_path,
        "clay_path": clay_path,
        "first_svg": bundle.sheets[0]["svg"] if bundle.sheets else "",
        "run_dir": str(run_dir),
        "true_north_deg": model_tn,
    }
    return (
        status_md,
        photoreal_path,
        clay_path,
        turntable_path,
        glb_path,
        sheets_html,
        downloads,
        ts_source,
        state,
        model_tn,
    )


def run_studio_pipeline(
    vibe_prompt: str,
    sketch_image,
    style_key: str,
    camera_key: str,
    spatial_rigor: float,
    mep_detail: float,
    clash_polish: float,
    seed: int = 42,
):
    run_dir = STUDIO_OUT / f"run_{int(time.time())}"
    fader_msg = apply_studio_faders(spatial_rigor, mep_detail, clash_polish)
    ts_source, bundle = generate_and_export_bim(vibe_prompt, sketch_image=sketch_image)
    return _build_visuals_from_bundle(
        ts_source, bundle, vibe_prompt, style_key, camera_key, fader_msg, run_dir, int(seed),
    )


def critique_and_redraw(
    state: dict,
    sketch_image,
    style_key: str,
    camera_key: str,
    spatial_rigor: float,
    mep_detail: float,
    clash_polish: float,
    seed: int = 42,
):
    if not state or not state.get("ts_source"):
        return ("Run a generation first before critiquing!", None, None, None, None, "", [], "", state, 0.0)

    run_dir = Path(state["run_dir"])
    critique_imgs = []
    if state.get("first_svg"):
        plan_png = svg_to_png(state["first_svg"], run_dir / "plan_critique.png")
        critique_imgs.append(Image.open(plan_png).convert("RGB"))
    if state.get("clay_path") and Path(state["clay_path"]).exists():
        critique_imgs.append(Image.open(state["clay_path"]).convert("RGB"))

    critique_prompt = (
        "Inspect the architectural floor plan and 3D cutaway of your previous MutinyBIM design. "
        "Identify any spatial circulation bottlenecks, awkward door placements, missing daylight windows, "
        "or cramped zones, then immediately output an improved TypeScript script.\n\n"
        f"Previous script:\n```ts\n{state['ts_source']}\n```"
    )
    fader_msg = apply_studio_faders(spatial_rigor, mep_detail, clash_polish)
    ts_source, bundle = generate_and_export_bim(
        state["vibe_prompt"],
        sketch_image=sketch_image,
        critique_context=critique_prompt,
        critique_images=critique_imgs,
    )
    new_dir = STUDIO_OUT / f"critique_{int(time.time())}"
    return _build_visuals_from_bundle(
        ts_source,
        bundle,
        state["vibe_prompt"],
        style_key,
        camera_key,
        fader_msg,
        new_dir,
        int(seed) + 1,
        banner_prefix="👁️ Vision Critique Redraw",
    )


SHOWCASE_CHOICES = [
    "small-office",
    "rooftop-unit",
    "plumbing-riser",
    "electrical-room",
    *[f"sketch:{t}" for t in SKETCH_TOPOLOGIES],
]


def run_instant_showcase(showcase_choice: str, style_key: str, camera_key: str, seed: int = 42):
    run_dir = STUDIO_OUT / f"showcase_{int(time.time())}"
    if showcase_choice.startswith("sketch:"):
        topo = showcase_choice.split(":", 1)[1]
        sample = generate_sketch_sample(int(seed) + 1000, topology=topo)
        ts_source = sample.ts_source
        vibe_prompt = sample.user_prompt
    else:
        ex = OFFICIAL_EXAMPLES.get(showcase_choice) or next(iter(OFFICIAL_EXAMPLES.values()))
        ts_source = ex["source"]
        vibe_prompt = ex["builds"]

    bundle = client.export_bundle(ts_source, project_name=f"Showcase {showcase_choice}", start="empty")
    return _build_visuals_from_bundle(
        ts_source,
        bundle,
        vibe_prompt,
        style_key,
        camera_key,
        f"⚡ Instant Showcase ({showcase_choice}, zero LLM wait)",
        run_dir,
        int(seed),
        banner_prefix="⚡ Showcase",
    )


def drop_random_napkin_sketch(seed: int = 42):
    s = int(seed) + int(time.time()) % 500
    topo = SKETCH_TOPOLOGIES[s % len(SKETCH_TOPOLOGIES)]
    style = SKETCH_STYLES[s % len(SKETCH_STYLES)]
    sample = generate_sketch_sample(s, topology=topo)
    out_png = STUDIO_OUT / f"napkin_{sample.sample_id}.png"
    render_parametric_sketch_png(sample, out_png, style=style)
    return Image.open(out_png).convert("RGB"), sample.user_prompt


def run_mixtape_strip(state: dict, mixtape_name: str, sketch_image, camera_key: str, seed: int = 42):
    preset = MIXTAPE_PRESETS.get(mixtape_name)
    if not preset:
        return None, f"Unknown mixtape preset: {mixtape_name}"

    mix_dir = STUDIO_OUT / f"mixtape_{int(time.time())}"
    mix_dir.mkdir(parents=True, exist_ok=True)
    mode = preset["mode"]
    tracks = preset["tracks"]

    if mode == "style_trio":
        if not state or not state.get("depth_path"):
            return None, "Generate or load a showcase building first, then drop the 3-track mixtape!"
        shot_paths, captions = [], []
        for idx, tr in enumerate(tracks):
            out_img = render_photoreal(
                state["depth_path"],
                state.get("vibe_prompt", ""),
                tr["style"],
                camera_key,
                mix_dir / f"track_{idx}.png",
                backend=RENDER_BACKEND,
                steps=max(16, RENDER_STEPS - 4),
                seed=int(seed),
            )
            shot_paths.append(out_img)
            captions.append(tr["label"])
        strip_path = compose_grid(shot_paths, captions, mix_dir / "mixtape_strip.png", cols=3)
        return strip_path, f"🎛️ Rendered **{mixtape_name}** locked to the exact same IFC depth geometry."

    # weight_sweep mode
    vibe_prompt = (state or {}).get("vibe_prompt") or VIBE_INSPIRATION_CHIPS[0]["prompt"]
    shot_paths, captions = [], []
    notes = [f"### 🎛️ {mixtape_name}"]
    for idx, tr in enumerate(tracks):
        sp, mep, cl = tr["faders"]
        f_msg = apply_studio_faders(sp, mep, cl)
        ts_src, bnd = generate_and_export_bim(vibe_prompt, sketch_image=sketch_image)
        t_dir = mix_dir / f"track_{idx}"
        t_dir.mkdir(parents=True, exist_ok=True)
        if bnd.ifc_text:
            ifc_f = t_dir / "model.ifc"
            ifc_f.write_text(bnd.ifc_text, encoding="utf-8")
            geom = ifc_to_glb_and_control(ifc_f, t_dir, camera_preset=camera_key, width=768, height=576)
            img_p = render_photoreal(
                geom["depth_path"], vibe_prompt, "ArchDaily Golden Hour Timber", camera_key, t_dir / "render.png",
                backend=RENDER_BACKEND, steps=max(16, RENDER_STEPS - 4), seed=int(seed),
            )
        else:
            img_p = str(t_dir / "fail.png")
            Image.new("RGB", (768, 576), (245, 235, 235)).save(img_p)
        shot_paths.append(img_p)
        lint_s = float(bnd.verify.breakdown.get("bim_lint_clean", 0.0))
        captions.append(f"{tr['label']} | Score {bnd.verify.reward:.2f} / Lint {lint_s:.2f}")
        notes.append(f"- **{tr['label']}**: {f_msg} → Overall `{bnd.verify.reward:.2f}`, Lint `{lint_s:.2f}`")
    strip_path = compose_grid(shot_paths, captions, mix_dir / "mixtape_strip.png", cols=3)
    return strip_path, "\n".join(notes)


def run_storyboard_board(state: dict, style_key: str, seed: int = 42, photoreal_all: bool = True):
    if not state or not state.get("ifc_path") or not Path(state["ifc_path"]).exists():
        return None, "Generate or load a showcase building with IFC geometry first!"
    board_dir = Path(state["run_dir"]) / f"storyboard_{int(time.time())}"
    res = render_storyboard_4up(
        state["ifc_path"],
        board_dir,
        state.get("vibe_prompt", ""),
        style_key,
        render_photoreal_shots=bool(photoreal_all),
        backend=RENDER_BACKEND,
        steps=max(16, RENDER_STEPS - 4),
        seed=int(seed),
    )
    mode_txt = "4 photoreal angles" if photoreal_all else "4 fast 3D clay angles"
    return res["board_path"], f"🎬 Rendered 2×2 architectural presentation board ({mode_txt}) from `model.ifc`."


def on_solar_city_change(city_key: str, date_key: str, hour_solar: float, true_north_deg: float):
    city = CITY_PRESETS.get(city_key) or next(iter(CITY_PRESETS.values()))
    lat_deg = float(city["lat_deg"])
    dial_svg = render_sun_dial_svg(
        lat_deg,
        KEY_DATES[date_key]["day_of_year"],
        float(hour_solar),
        city_label=city["label"],
        date_label=KEY_DATES[date_key]["label"],
        true_north_deg=float(true_north_deg),
    )
    return lat_deg, dial_svg


def update_sun_dial_ui(city_key: str, lat_deg: float, date_key: str, hour_solar: float, true_north_deg: float) -> str:
    city = CITY_PRESETS.get(city_key) or next(iter(CITY_PRESETS.values()))
    dt = KEY_DATES.get(date_key) or next(iter(KEY_DATES.values()))
    return render_sun_dial_svg(
        float(lat_deg),
        int(dt["day_of_year"]),
        float(hour_solar),
        city_label=city["label"] if abs(float(lat_deg) - city["lat_deg"]) < 0.02 else f"Lat {float(lat_deg):.2f}°",
        date_label=dt["label"],
        true_north_deg=float(true_north_deg),
    )


def run_solar_study(
    state: dict,
    city_key: str,
    lat_deg: float,
    date_key: str,
    hour_solar: float,
    true_north_deg: float,
    grid_spacing_m: float,
):
    dial_html = update_sun_dial_ui(city_key, lat_deg, date_key, hour_solar, true_north_deg)
    if not state or not state.get("ifc_path") or not Path(state["ifc_path"]).exists():
        return None, None, dial_html, "Generate or load a showcase building with IFC geometry first!", None

    dt = KEY_DATES.get(date_key) or next(iter(KEY_DATES.values()))
    doy = int(dt["day_of_year"])
    solar_dir = Path(state["run_dir"]) / f"solar_{date_key}_{int(time.time())}"
    solar_dir.mkdir(parents=True, exist_ok=True)

    res = compute_direct_sun_hours(
        state["ifc_path"],
        lat_deg=float(lat_deg),
        day_of_year=doy,
        true_north_deg=float(true_north_deg),
        step_minutes=30,
        grid_spacing_m=float(grid_spacing_m),
    )
    dome_path = render_solar_dome_png(
        res,
        solar_dir / "solar_dome.png",
        lat_deg=float(lat_deg),
        day_of_year=doy,
        true_north_deg=float(true_north_deg),
        title=f"Direct Sun Hours — {dt['label']}",
    )
    shadow_gif = render_solstice_shadow_gif(
        state["ifc_path"],
        solar_dir / "solar_shadows.gif",
        lat_deg=float(lat_deg),
        day_of_year=doy,
        true_north_deg=float(true_north_deg),
    )
    heatmap_path = solar_dir / "solar_heatmap.json"
    heatmap_path.write_text(json.dumps(res["overlay"], indent=2), encoding="utf-8")

    summary_md = (
        f"### ☀️ Passive-Solar Exposure (Direct Sun Hours) — {dt['label']}\n"
        f"- **Mean:** `{res['mean_hours']:.2f} h` · **Max:** `{res['max_hours']:.2f} h` · "
        f"**Min:** `{res['min_hours']:.2f} h`\n"
        f"- **Area ≥ 4 h Direct Sun:** `{res['pct_over_4h']:.1f}%` · **Workplane Z:** `{res['workplane_z_m']:.2f} m` · "
        f"**True North:** `{res['true_north_deg']:+.1f}°` · **Grid Cells:** `{len(res['cells'])}`\n\n"
        f"> *{res.get('assumptions', '')}*"
    )
    return dome_path, shadow_gif, dial_html, summary_md, str(heatmap_path)


def run_shadow_locked_photoreal_hour(
    state: dict,
    city_key: str,
    lat_deg: float,
    date_key: str,
    hour_solar: float,
    true_north_deg: float,
    style_key: str,
    camera_key: str,
    shadow_strength: float,
    seed: int,
    photoreal_enabled: bool,
):
    if not state or not state.get("ifc_path") or not Path(state["ifc_path"]).exists():
        return None, None, "Generate or load a showcase building with IFC geometry first!"

    dt = KEY_DATES.get(date_key) or next(iter(KEY_DATES.values()))
    doy = int(dt["day_of_year"])
    out_dir = Path(state["run_dir"]) / f"golden_{date_key}_{int(time.time())}"
    ctrl = render_solar_control_frame(
        state["ifc_path"],
        out_dir,
        lat_deg=float(lat_deg),
        day_of_year=doy,
        hour_solar=float(hour_solar),
        true_north_deg=float(true_north_deg),
        camera_preset=camera_key,
    )
    lighting = ctrl["lighting"]
    if photoreal_enabled:
        hero_path = render_photoreal(
            ctrl["depth_path"],
            state.get("vibe_prompt", ""),
            style_key,
            camera_key,
            out_dir / "hero_photoreal.png",
            backend=RENDER_BACKEND,
            steps=RENDER_STEPS,
            seed=int(seed),
            shadow_init_path=ctrl["shadow_clay_path"],
            shadow_strength=float(shadow_strength),
            solar_lighting_cue=lighting["prompt_cue"],
            solar_negative_cue=lighting["negative_cue"],
        )
    else:
        hero_path = ctrl["shadow_clay_path"]

    notes = (
        f"### 🌅 Shadow-Locked Photoreal ({dt['label']}, {hour_solar:.1f} h solar)\n"
        f"- **Phase:** `{lighting['phase']}` · **Alt:** `{lighting['altitude_deg']:.1f}°` · "
        f"**Az:** `{lighting['azimuth_deg']:.1f}°` · **Direction in view:** `{lighting['screen_dir']}`\n"
        f"- **Lighting prompt cue:** *{lighting['prompt_cue']}*"
    )
    if ctrl["roof_assumed"]:
        notes += "\n- *Roof assumed for shadow projection (model lacks IfcRoof / top IfcSlab).*"
    if ctrl["camera_fallback"]:
        notes += f"\n- ⚠️ Camera fallback (`{ctrl['fallback_reason']}`)"
    return hero_path, ctrl["shadow_clay_path"], notes


def run_golden_hour_timelapse(
    state: dict,
    city_key: str,
    lat_deg: float,
    date_key: str,
    true_north_deg: float,
    style_key: str,
    camera_key: str,
    shadow_strength: float,
    seed: int,
    photoreal_enabled: bool,
):
    if not state or not state.get("ifc_path") or not Path(state["ifc_path"]).exists():
        return None, None, "Generate or load a showcase building with IFC geometry first!"

    dt = KEY_DATES.get(date_key) or next(iter(KEY_DATES.values()))
    doy = int(dt["day_of_year"])
    out_dir = Path(state["run_dir"]) / f"timelapse_{date_key}_{int(time.time())}"
    res = render_solar_photoreal_timelapse(
        state["ifc_path"],
        out_dir,
        state.get("vibe_prompt", ""),
        style_key,
        lat_deg=float(lat_deg),
        day_of_year=doy,
        true_north_deg=float(true_north_deg),
        camera_preset=camera_key,
        num_frames=4,
        render_photoreal_frames=bool(photoreal_enabled),
        backend=RENDER_BACKEND,
        steps=max(16, RENDER_STEPS - 4),
        seed=int(seed),
        shadow_strength=float(shadow_strength),
    )
    hrs_str = ", ".join(f"{h:.1f}h" for h in res["hours_solar"])
    notes = (
        f"### 🎞️ Golden-Hour Photoreal Time-Lapse ({dt['label']})\n"
        f"- **Sampled solar hours:** `{hrs_str}` · **Roof assumed:** `{res['roof_assumed']}`"
    )
    return res["board_path"], res["gif_path"], notes


RADIATION_PERIODS = {
    "Annual (Jan 1 – Dec 31)": (1, 365, 3),
    "Summer Cooling (Jun 1 – Aug 31)": (152, 243, 1),
    "Winter Heating (Dec 1 – Feb 28)": (335, 59, 1),
    "Selected Key Date (1-Day)": None,
}


def run_incident_radiation_study(
    state: dict,
    city_key: str,
    lat_deg: float,
    date_key: str,
    true_north_deg: float,
    period_label: str,
    epw_file,
):
    if not state or not state.get("ifc_path") or not Path(state["ifc_path"]).exists():
        return None, "Generate or load a showcase building with IFC geometry first!", None

    dt = KEY_DATES.get(date_key) or next(iter(KEY_DATES.values()))
    p_spec = RADIATION_PERIODS.get(period_label)
    if p_spec is None:
        doy = int(dt["day_of_year"])
        start_doy, end_doy, day_step = doy, doy, 1
    else:
        start_doy, end_doy, day_step = p_spec

    epw_path = None
    if epw_file is not None:
        epw_path = str(getattr(epw_file, "name", epw_file))

    rad_dir = Path(state["run_dir"]) / f"radiation_{int(time.time())}"
    rad_dir.mkdir(parents=True, exist_ok=True)
    res = compute_incident_solar_radiation(
        state["ifc_path"],
        lat_deg=None if (epw_path and abs(float(lat_deg) - float((CITY_PRESETS.get(city_key) or {}).get("lat_deg", -999))) < 0.02) else float(lat_deg),
        start_doy=start_doy,
        end_doy=end_doy,
        true_north_deg=float(true_north_deg),
        epw_path_or_text=epw_path,
        day_step=1 if epw_path else day_step,
    )
    rad_png = render_incident_radiation_png(
        res,
        rad_dir / "incident_radiation.png",
        title=f"Incident Radiation (kWh/m²) — {period_label}",
    )
    rad_json = rad_dir / "radiation_heatmap.json"
    rad_json.write_text(json.dumps(res["overlay"], indent=2), encoding="utf-8")

    by_cls_lines = " · ".join(
        f"**{k}**: `{v['mean_kwh_m2']:.1f} kWh/m²` ({v['area_m2']:.1f} m²)"
        for k, v in res["by_class"].items()
    )
    md = (
        f"### 🔥 Cumulative Incident Solar Radiation (`kWh/m²`) — {period_label}\n"
        f"- **Sky source:** `{res['sky_source']}` · **Mean:** `{res['mean_kwh_m2']:.1f} kWh/m²` · "
        f"**Max:** `{res['max_kwh_m2']:.1f} kWh/m²` · **Total Surface Area:** `{res['total_surface_area_m2']:.1f} m²` · "
        f"**Total Incident Energy:** `{res['total_incident_kwh']:.0f} kWh`\n"
        f"- **By element class:** {by_cls_lines}\n\n"
        f"> *{res['assumptions']}*"
    )
    return rad_png, md, str(rad_json) 


In [ ]:
#@title 6. Launch Vibe Architect Studio — Mixtape Edition (Gradio Web UI)
import gradio as gr

with gr.Blocks(title="MutinyBIM Vibe Architect Studio", theme=gr.themes.Soft()) as studio:
    gr.Markdown(
        "# 🏗️ MutinyBIM Vibe Architect Studio — Mixtape Edition\n"
        "Drop a napkin sketch or prompt (or load an **Instant Showcase**) → get a **real parametric IFC 4.3 building**, "
        "**printable architectural sheets**, **360° turntable GIF**, **Mixtape A/B strip**, and a **4-shot photoreal storyboard**."
    )
    studio_state = gr.State({})

    with gr.Row():
        with gr.Column(scale=4):
            sketch_in = gr.Image(label="Napkin Sketch / Floor Plan Screenshot (Optional)", type="pil", height=210)
            drop_sketch_btn = gr.Button("🎲 Drop Synthetic Napkin Sketch", size="sm")

            vibe_in = gr.Textbox(
                label="Vibe Prompt",
                lines=4,
                value=VIBE_INSPIRATION_CHIPS[0]["prompt"],
                placeholder="Describe your space, materials, rooms, daylighting, or MEP vibe...",
            )
            with gr.Row():
                for chip in VIBE_INSPIRATION_CHIPS:
                    btn = gr.Button(chip["title"], size="sm")
                    btn.click(fn=lambda p=chip["prompt"]: p, outputs=vibe_in)

            style_in = gr.Dropdown(
                label="Art Direction",
                choices=list(ART_DIRECTION_PRESETS.keys()),
                value="ArchDaily Golden Hour Timber",
            )
            camera_in = gr.Radio(
                label="Camera Angle",
                choices=[(v["label"], k) for k, v in CAMERA_PRESETS.items()],
                value="isometric_cutaway",
            )
            seed_in = gr.Number(label="Render Seed", value=42, precision=0)

            with gr.Accordion("⚡ Instant Showcase (Zero LLM Wait)", open=False):
                showcase_dd = gr.Dropdown(
                    label="Pick a verified MutinyBIM example or parametric sketch topology",
                    choices=SHOWCASE_CHOICES,
                    value="small-office",
                )
                showcase_btn = gr.Button("⚡ Load Showcase Building", size="sm")

            with gr.Accordion("🎛️ Design Mixing Desk (Weight-Space Faders)", open=False):
                spatial_slider = gr.Slider(0.0, 1.5, value=0.9, step=0.1, label="Spatial & Architectural Rigor")
                mep_slider = gr.Slider(0.0, 1.5, value=0.7, step=0.1, label="MEP & Code Engineering Detail")
                clash_slider = gr.Slider(0.0, 1.0, value=0.3, step=0.1, label="Clash-Negation Polish (-λ)")

            with gr.Row():
                gen_btn = gr.Button("✨ Generate Real Building", variant="primary", scale=2)
                critique_btn = gr.Button("👁️ Critique & Redraw", variant="secondary", scale=1)

        with gr.Column(scale=6):
            status_out = gr.Markdown("Ready — click **✨ Generate Real Building** or **⚡ Load Showcase Building**.")
            with gr.Tabs():
                with gr.Tab("🎨 Photoreal Render"):
                    photoreal_out = gr.Image(label="Geometry-Locked Photoreal Render", type="filepath")
                    clay_out = gr.Image(label="True BIM 3D Clay / Geometry Reference", type="filepath")
                with gr.Tab("🎛️ Mixtape A/B Strip"):
                    mixtape_dd = gr.Dropdown(
                        label="Mixtape Recipe",
                        choices=list(MIXTAPE_PRESETS.keys()),
                        value=list(MIXTAPE_PRESETS.keys())[0],
                    )
                    mixtape_btn = gr.Button("🔥 Drop 3-Track Mixtape", variant="primary")
                    mixtape_strip_out = gr.Image(label="3-Track Comparison Triptych", type="filepath")
                    mixtape_notes_out = gr.Markdown("")
                with gr.Tab("🎬 4-Shot Board & 360° Orbit"):
                    turntable_out = gr.Image(label="360° Isometric Cutaway Orbit (Live IFC Geometry)", type="filepath")
                    with gr.Row():
                        photoreal_board_chk = gr.Checkbox(label="Photoreal Render All 4 Angles (unchecked = instant 3D clay)", value=True)
                        storyboard_btn = gr.Button("🎬 Render 4-Shot Storyboard", variant="primary")
                    storyboard_out = gr.Image(label="2x2 Architectural Presentation Board", type="filepath")
                    storyboard_notes_out = gr.Markdown("")
                with gr.Tab("🧊 Interactive 3D Model"):
                    glb_out = gr.Model3D(label="Interactive IFC → GLB Model", height=520)
                with gr.Tab("📐 Architectural Plan Sheets"):
                    sheets_out = gr.HTML(label="MutinyBIM Vector SVG Sheets")
                with gr.Tab("📦 Deliverables & Code"):
                    files_out = gr.Files(label="Download IFC 4.3, Vector PDF, 3D GLB, Turntable GIF & Script")
                    code_out = gr.Code(label="Generated @permitzip/bim-script", language="typescript")
                with gr.Tab("☀️ Solar Dome & Sun Hours"):
                    with gr.Row():
                        solar_city_dd = gr.Dropdown(
                            label="Location Preset",
                            choices=[(v["label"], k) for k, v in CITY_PRESETS.items()],
                            value="san_francisco",
                        )
                        solar_lat_num = gr.Number(label="Latitude (°)", value=37.7749, precision=4)
                        solar_date_dd = gr.Dropdown(
                            label="Key Solar Date",
                            choices=[(v["label"], k) for k, v in KEY_DATES.items()],
                            value="summer_solstice",
                        )
                    with gr.Row():
                        solar_hour_slider = gr.Slider(
                            6.0, 18.0, value=12.0, step=0.5,
                            label="Solar Time (local solar hr, ±1° approx., not clock time)",
                        )
                        solar_north_slider = gr.Slider(
                            -180.0, 180.0, value=0.0, step=5.0,
                            label="True North (° CW from plan north — auto-read from IFC, drag to override)",
                        )
                        solar_spacing_slider = gr.Slider(0.5, 2.0, value=1.0, step=0.25, label="Heatmap Grid Spacing (m)")
                    with gr.Row():
                        solar_dial_html = gr.HTML(
                            value=render_sun_dial_svg(
                                37.7749,
                                172,
                                12.0,
                                city_label=CITY_PRESETS["san_francisco"]["label"],
                                date_label=KEY_DATES["summer_solstice"]["label"],
                            )
                        )
                        with gr.Column():
                            run_solar_btn = gr.Button("☀️ Run Direct Sun Hours & Sky Dome", variant="primary")
                            solar_notes_out = gr.Markdown(
                                "Select city & date above, then run Direct Sun Hours on the active IFC building."
                            )
                            solar_json_out = gr.File(
                                label="Download JSON (MutinyBIM ScalarHeatmapOverlay format — requires editor add-on calling setScalarHeatmap)"
                            )
                    with gr.Row():
                        solar_dome_out = gr.Image(label="Ladybug 3D Solar Dome & Workplane Direct Sun Hours", type="filepath")
                        solar_shadow_gif_out = gr.Image(label="Hourly Shadow Sweep Animation", type="filepath")
                    gr.Markdown("### 🌅 Shadow-Conditioned Photoreal Golden Hour (ControlNet-Depth + Analytical Shadow Clay)")
                    with gr.Row():
                        solar_strength_slider = gr.Slider(
                            0.55, 0.90, value=0.76, step=0.02,
                            label="Shadow-Clay Init Strength (lower = stricter shadow lock, higher = richer photoreal detail)",
                        )
                        solar_photoreal_chk = gr.Checkbox(
                            label="Run Photoreal Diffusion (uncheck for instant analytical shadow clay)",
                            value=True,
                        )
                    with gr.Row():
                        solar_hero_btn = gr.Button("🌅 Render Selected Hour (Shadow-Locked Photoreal)", variant="primary")
                        solar_timelapse_btn = gr.Button("🎞️ Render 4-Hour Golden-Hour Time-Lapse", variant="secondary")
                    with gr.Row():
                        solar_hero_out = gr.Image(label="Shadow-Locked Photoreal / 4-Hour Board", type="filepath")
                        solar_clay_guide_out = gr.Image(label="Physical Shadow-Clay Guide", type="filepath")
                        solar_timelapse_gif_out = gr.Image(label="Golden-Hour Time-Lapse GIF", type="filepath")
                    solar_golden_notes_out = gr.Markdown("")
                    gr.Markdown("### 🔥 Cumulative Incident Solar Radiation (kWh/m² — Ladybug Tregenza 145-Patch)")
                    with gr.Row():
                        rad_period_dd = gr.Dropdown(
                            label="Accumulation Period",
                            choices=list(RADIATION_PERIODS.keys()),
                            value="Annual (Jan 1 – Dec 31)",
                        )
                        rad_epw_file = gr.File(
                            label="Optional EnergyPlus Weather File (.epw — uses clear-sky model when empty)",
                            file_types=[".epw"],
                        )
                        rad_btn = gr.Button("🔥 Run Incident Radiation (kWh/m²)", variant="primary")
                    with gr.Row():
                        rad_png_out = gr.Image(label="3D Surface Incident Radiation (kWh/m²)", type="filepath")
                        with gr.Column():
                            rad_notes_out = gr.Markdown("Run cumulative kWh/m² surface radiation across IFC walls, roofs, and windows.")
                            rad_json_out = gr.File(label="Download Surface Radiation JSON (kWh/m²)")

    inputs = [vibe_in, sketch_in, style_in, camera_in, spatial_slider, mep_slider, clash_slider, seed_in]
    outputs = [status_out, photoreal_out, clay_out, turntable_out, glb_out, sheets_out, files_out, code_out, studio_state, solar_north_slider]

    gen_btn.click(fn=run_studio_pipeline, inputs=inputs, outputs=outputs)
    critique_btn.click(
        fn=critique_and_redraw,
        inputs=[studio_state, sketch_in, style_in, camera_in, spatial_slider, mep_slider, clash_slider, seed_in],
        outputs=outputs,
    )
    showcase_btn.click(
        fn=run_instant_showcase,
        inputs=[showcase_dd, style_in, camera_in, seed_in],
        outputs=outputs,
    )
    drop_sketch_btn.click(
        fn=drop_random_napkin_sketch,
        inputs=[seed_in],
        outputs=[sketch_in, vibe_in],
    )
    mixtape_btn.click(
        fn=run_mixtape_strip,
        inputs=[studio_state, mixtape_dd, sketch_in, camera_in, seed_in],
        outputs=[mixtape_strip_out, mixtape_notes_out],
    )
    storyboard_btn.click(
        fn=run_storyboard_board,
        inputs=[studio_state, style_in, seed_in, photoreal_board_chk],
        outputs=[storyboard_out, storyboard_notes_out],
    )
    solar_city_dd.change(
        fn=on_solar_city_change,
        inputs=[solar_city_dd, solar_date_dd, solar_hour_slider, solar_north_slider],
        outputs=[solar_lat_num, solar_dial_html],
    )
    for comp in (solar_lat_num, solar_date_dd, solar_hour_slider, solar_north_slider):
        comp.change(
            fn=update_sun_dial_ui,
            inputs=[solar_city_dd, solar_lat_num, solar_date_dd, solar_hour_slider, solar_north_slider],
            outputs=solar_dial_html,
        )
    run_solar_btn.click(
        fn=run_solar_study,
        inputs=[
            studio_state,
            solar_city_dd,
            solar_lat_num,
            solar_date_dd,
            solar_hour_slider,
            solar_north_slider,
            solar_spacing_slider,
        ],
        outputs=[solar_dome_out, solar_shadow_gif_out, solar_dial_html, solar_notes_out, solar_json_out],
    )
    solar_hero_btn.click(
        fn=run_shadow_locked_photoreal_hour,
        inputs=[
            studio_state,
            solar_city_dd,
            solar_lat_num,
            solar_date_dd,
            solar_hour_slider,
            solar_north_slider,
            style_in,
            camera_in,
            solar_strength_slider,
            seed_in,
            solar_photoreal_chk,
        ],
        outputs=[solar_hero_out, solar_clay_guide_out, solar_golden_notes_out],
    )
    solar_timelapse_btn.click(
        fn=run_golden_hour_timelapse,
        inputs=[
            studio_state,
            solar_city_dd,
            solar_lat_num,
            solar_date_dd,
            solar_north_slider,
            style_in,
            camera_in,
            solar_strength_slider,
            seed_in,
            solar_photoreal_chk,
        ],
        outputs=[solar_hero_out, solar_timelapse_gif_out, solar_golden_notes_out],
    )
    rad_btn.click(
        fn=run_incident_radiation_study,
        inputs=[
            studio_state,
            solar_city_dd,
            solar_lat_num,
            solar_date_dd,
            solar_north_slider,
            rad_period_dd,
            rad_epw_file,
        ],
        outputs=[rad_png_out, rad_notes_out, rad_json_out],
    )

studio.queue().launch(share=SHARE_PUBLIC_GRADIO_LINK, debug=False)
